# README
- **Author**: RUEI-HSIANG, CHANG（張睿庠；B12303030；AI 協作）
- **Created At**: 2026-10-18
- **Last Modified At**: 2026-10-18

## What does this file do?
依使用者「都抓」指示整理2026議員登記名冊、內政部現任議員名錄及第11屆立委名錄；三類分表，不合併為研究樣本。保留原始姓名與推薦政黨，只另欄移除排版換行。未列任期不推定起訖；未核實的社群帳號不記為不存在，失敗存取不填零貼文。

## What does this file take?
- **Source Data Sets**: 下方固定 INPUT_HASHES 逐檔列明；原檔見 [來源文件](../raw/online_20261018_documentation.ipynb)。
- PDF→JSON 由 `extract_council_registration.py` 產生（需pdfplumber），原PDF不變；本Notebook依固定JSON版本整理。
- **Input key**: CEC為文件／頁／列；MOI為_PARENT_ID；立院為nodeid；既有候選人為candidate_key。

## What does this file output?
- `data/processed/council_candidates_2026_registration.csv`：候選人登記，一列一登記；key=candidate_key。
- `data/processed/current_councilors_20261018.csv`：內政部名錄一列一人；key=official_key。
- `data/processed/legislators_term11_directory_20261018.csv`：第11屆一列一人；key=legislator_key，現任與離職分欄。
- `data/processed/meta_candidate_account_worklist_20261018.csv`：一列一候選人×平台；不是貼文資料。
- `data/processed/meta_access_attempts_20261018.csv`：一列一次存取；不是廣告或社群觀察。
- **Role**: preparation/source-directory/access-audit，均非已完成分析樣本。
- **Documentation**: [整理後文件](../processed/online_20261018_documentation.ipynb)。
- **Expected schema and row-count rules**: CEC兩份PDF共1502列、22縣市；MOI鍵唯一；立院current=113、former另列；工作單81×3；重讀逐值相等。
- **Verification status / SHA-256**: 執行輸出逐檔列示；後續文件固定讀取同一hash。
- **Downstream finding input**: 以上明確路徑；正式樣本納入條件仍待確認。


In [1]:
from pathlib import Path
import hashlib, json, re
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs
from IPython.display import display
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'AGENTS.md').is_file() and (p/'templates').is_dir())

INPUT_HASHES = {'data/raw/online_20261018/capture_manifest.json': '3add11cf37c58de398b6a9c54e384d80fb8c23911d28cce021a3886652f6fa3f', 'data/raw/online_20261018/cec_council_county.pdf': '11d6334e4a5cf3bc7539455ef047ac40f62e250d3c151348683ab1141f83d517', 'data/raw/online_20261018/cec_council_county_tables.json': '04e15c58f7e6af08dfada94ee643bb94fe4610833e5c471f41c050b7e36fce79', 'data/raw/online_20261018/cec_council_metropolitan.pdf': '28a9bf7edeb4b2f45a0b3ca473d59c930fcf8cba575c8879fb33397815116090', 'data/raw/online_20261018/cec_council_metropolitan_tables.json': '645a70d9312d41be408b1ba0579f12924f000e0b10b9d44f5f5b4047eb04bea2', 'data/raw/online_20261018/ly_members.html': '54202822c014d09a5e78fa68309906661345ae3e553fb1210d6b75192c77951a', 'data/raw/online_20261018/meta_access_01.html': 'da0d3c1a3eaf9fbd40707aeef61e4eee804017beff4006e9c3f048dd371dcc6d', 'data/raw/online_20261018/meta_access_02.html': '19e33623156a1cf3d10ba4ce835413c80c0a201a98b311c2eeac49a4a1cbceca', 'data/raw/online_20261018/meta_access_03.html': 'fc0b32adb923ca383f12d936331df5a6e259df1fa990b56c2f9946ab8af436ea', 'data/raw/online_20261018/meta_access_04.html': '6d6f3b5e5971aa42770ad0910ba16a9fd30fcca825c65e356777be8b3790fc6e', 'data/raw/online_20261018/meta_access_05.html': 'e6654daaf8515a41a17c6d96444dd436f366b35f7cf502b150fe9bdee30cde26', 'data/raw/online_20261018/meta_access_06.html': '500e367cd7c691907071116246e71842f8d9628c8f506dafb0f1c4876b95dda7', 'data/raw/online_20261018/meta_access_07.html': '166db55b42133547f09b02bb17f1e4832dbcb1e8555496cd4223a6d3a1871077', 'data/raw/online_20261018/meta_access_08.html': '4f607fed0095311033066f512f02b9d16fd5de4b660810a586fdefc556b259be', 'data/raw/online_20261018/meta_access_09.html': 'bbf55ba001f1f6d36df884aa37287dcd207db9a1dced3f4351eb04706673bc46', 'data/raw/online_20261018/meta_access_10.html': '86ec64ff936e0df7f72000abf0d05b08de7febac8471086ab5fa4e984b1af2d1', 'data/raw/online_20261018/meta_access_11.html': '0a7c8f615e6da11adcba6fbe04726a6193cf72d06db1f5834ae78c39919cada4', 'data/raw/online_20261018/meta_access_12.html': '9bf0f39436349994f5d3e2043154352f2440fe9b757df1cff4a66de8c6383691', 'data/raw/online_20261018/meta_access_13.html': '7cf17bc7d6755f450de95c24d4b061a85b058b96c4fcc2ac94517cbb383af939', 'data/raw/online_20261018/meta_access_manifest.json': 'b8069512d839d2708487303f8c6b08cdf43d2776032288299c5fdf09b736b93a', 'data/raw/online_20261018/moi_county.html': '7ba34c8d17af8cf5bad108312e3218e5f3e5444fef457966f19ec199b97b50b5', 'data/raw/online_20261018/moi_metropolitan.html': '41294443945db5603a7e664434b7532ff58fbc75f1ab63aa6d7258d06486c5af', 'data/processed/candidate_campaign_sources.csv': '682e1fffd9671ae2182548f07a0894b489d8edd1803f4b723106b991115341bd', 'data/processed/candidates_2026_registration.csv': '765797d09de8e70f6cb54486cf660c44f93e7aab1c68c3a6233db60aad7c5832'}
for rel, expected in INPUT_HASHES.items():
    p=(ROOT/rel).resolve()
    assert p.is_relative_to(ROOT) and p.is_file()
    assert hashlib.sha256(p.read_bytes()).hexdigest()==expected, rel
print('Verified inputs:',len(INPUT_HASHES))


Verified inputs: 24


In [ ]:
rows=[]
for stem,office in [('cec_council_metropolitan','直轄市議員'),('cec_council_county','縣市議員')]:
    pages=json.loads((ROOT/f'data/raw/online_20261018/{stem}_tables.json').read_text())
    src=next(x for x in json.loads((ROOT/'data/raw/online_20261018/capture_manifest.json').read_text()) if x['path'].endswith(stem+'.pdf'))
    for page in pages:
        assert len(page['tables'])==1
        table=page['tables'][0]
        assert table[0]==['選舉區','登記日期','姓名','推薦之政黨','備註']
        for district,date,name,party,note in table[1:]:
            assert district and date and name and party
            county=re.match(r'^(.+?[市縣])',district).group(1)
            clean_name=re.sub(r'\s+','',name)
            key=f'2026|{office}|{district}|{clean_name}'
            rows.append(dict(candidate_key=key,election_year='2026',office_type=office,county_city=county,electoral_district=district,registration_date_roc=date,candidate_name=clean_name,candidate_name_original=name,party_recommendation_original=party,party_recommendation_joined=re.sub(r'\s+','',party),source_note=note or '',source_file=src['path'],source_page=str(page['page']),source_url=src['url'],qualification_status='registration_only_not_final_qualification',captured_date='2026-10-18'))
council=pd.DataFrame(rows).fillna('')
# Flatten PDF layout line breaks only; raw table JSON retains original typography.
for col in council.columns:
    council[col] = council[col].map(lambda value: re.sub(r'[\r\n]+', ' ', value))
assert len(council)==1502 and council.candidate_key.is_unique and council.county_city.nunique()==22
current=[]
for file,office in [('moi_metropolitan.html','直轄市議員'),('moi_county.html','縣市議員')]:
    path=ROOT/'data/raw/online_20261018'/file
    soup=BeautifulSoup(path.read_text(),'html.parser')
    for block in soup.select('div.block'):
        caption=block.select_one('.essay .caption'); link=block.select_one('a[href*="LocalOfficial_Content"]')
        if caption is None or link is None: continue
        def text(sel):
            e=block.select_one(sel);return e.get_text(' ',strip=True) if e else ''
        url=urljoin('https://www.moi.gov.tw',link['href']);pid=parse_qs(urlparse(url).query)['_PARENT_ID'][0]
        current.append(dict(official_key=pid,official_name=text('.essay .caption'),county_city=text('.essay .locate'),organization_and_position=text('.essay .position'),party_original=text('.essay .group'),office_type=office,profile_url=url,source_file=str(path.relative_to(ROOT)),captured_date='2026-10-18',status='listed_in_moi_current_directory',term_start='',term_end='',term_note='名錄未列任期；不以2022當選自動推定持續在任'))
current=pd.DataFrame(current).fillna('')
assert current.official_key.is_unique and current.county_city.nunique()==22
ly=[]
soup=BeautifulSoup((ROOT/'data/raw/online_20261018/ly_members.html').read_text(),'html.parser')
for block in soup.select('.six-legislatorAvatar'):
    link=block.select_one('a[href]');name=block.select_one('.legislatorname').get_text(' ',strip=True)
    header=block.find_previous('h2').get_text(' ',strip=True)
    state='former' if '離職' in header else 'current'
    image=block.select_one('.six-party-icon');image_url=image.get('src','') if image else ''
    # Preserve party logo URL as provenance, do not infer party from image pixels.
    ly.append(dict(legislator_key=parse_qs(urlparse(link['href']).query)['nodeid'][0],name=name,term='11',membership_status=state,party_logo_url=urljoin('https://www.ly.gov.tw',image_url),profile_url=urljoin('https://www.ly.gov.tw',link['href']),source_file='data/raw/online_20261018/ly_members.html',captured_date='2026-10-18'))
ly=pd.DataFrame(ly).fillna('')
assert ly.legislator_key.is_unique and (ly.membership_status=='current').sum()==113
sources=pd.read_csv(ROOT/'data/processed/candidate_campaign_sources.csv',dtype=str,keep_default_na=False)
reg=pd.read_csv(ROOT/'data/processed/candidates_2026_registration.csv',dtype=str,keep_default_na=False)
work=[]
for _,r in reg.iterrows():
    for platform in ['Facebook','Instagram','Threads']:
        matches=sources[(sources.candidate_key==r.candidate_key)&(sources.source_kind=={'Facebook':'facebook.com','Instagram':'instagram.com','Threads':'threads.net'}[platform])]
        urls=sorted(set(matches.url))
        work.append(dict(candidate_key=r.candidate_key,candidate_name=r.candidate_name,platform=platform,official_urls=';'.join(urls),discovered_on=';'.join(sorted(set(matches.discovered_on))),account_status='official_link_previously_saved' if urls else 'not_verified_not_absent',posts_status='not_collected_not_zero',window_count='',followers_start='',followers_end='',updated_date='2026-10-18'))
work=pd.DataFrame(work)
assert len(work)==243 and not work.duplicated(['candidate_key','platform']).any()
access=pd.DataFrame(json.loads((ROOT/'data/raw/online_20261018/meta_access_manifest.json').read_text())).fillna('')
for idx,row in access.iterrows():
    p=ROOT/row['path'] if row['path'] else None
    soup=BeautifulSoup(p.read_bytes(),'html.parser') if p and p.is_file() else None
    title=soup.title.get_text(' ',strip=True) if soup and soup.title else ''
    visible=soup.get_text(' ',strip=True) if soup else ''
    access.loc[idx,'response_title']=title
    access.loc[idx,'visible_text_chars']=str(len(visible))
    access.loc[idx,'content_status']=('access_challenge' if row['http_status']!='200' else 'public_documentation' if row['subject'] in ['Meta research announcement','Ad Library API'] else 'page_shell_without_verified_posts')

outputs={'council_candidates_2026_registration':council,'current_councilors_20261018':current,'legislators_term11_directory_20261018':ly,'meta_candidate_account_worklist_20261018':work,'meta_access_attempts_20261018':access}
checks=[]
for stem,df in outputs.items():
    path=ROOT/'data/processed'/(stem+'.csv')
    assert path.resolve().is_relative_to((ROOT/'data/processed').resolve())
    df.to_csv(path,index=False,encoding='utf-8-sig')
    read=pd.read_csv(path,dtype=str,keep_default_na=False)
    expected=df.astype(str)
    pd.testing.assert_frame_equal(read,expected)
    checks.append(dict(path=str(path.relative_to(ROOT)),rows=len(read),columns=len(read.columns),sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
display(pd.DataFrame(checks))
display(council.groupby(['office_type','county_city']).size().rename('registrations').to_frame())
display(current.groupby('office_type').size().rename('source_listed_members').to_frame())
display(ly.membership_status.value_counts().to_frame())
display(work.groupby(['platform','account_status']).size().rename('rows').to_frame())
display(access[['subject','http_status','return_code','interpretation']])


2026-10-20 維護：重複檔案改用單一保存路徑，固定指紋同步更新；議員登記CSV排版換行改為空格，政黨標籤不變。本次已獨立從raw表格重建並比對全部1,502筆及16欄；未重跑本Notebook或整條研究流程，既有執行輸出屬先前執行紀錄。